In [ ]:
#
# Importación de los datos
#
import pandas as pd  #  type: ignore

df = pd.read_csv("../data/mtcars.csv")
df.head()

In [ ]:
# Se verifica el tamaño del conjunto de datos.

df.shape

In [ ]:
# Se identifican valores nulos antes de entrenar el modelo.

df.isnull().sum()

In [ ]:
#
# Correlación entre variables
#
import seaborn as sns
import matplotlib.pyplot as plt

corr = df.corr()
plt.figure(figsize=(10, 6))
sns.heatmap(
    corr,
    xticklabels=corr.columns.values,
    yticklabels=corr.columns.values,
    cmap="BuPu",
    vmin=-1,
    vmax=1,
    annot=True,
)
plt.title("Correlation Heatmap of mtcars dataset")
plt.show()

In [ ]:
#
# Particionamiento de los datos
#
from sklearn.model_selection import train_test_split

features = df.columns[1:]
target = df.columns[0]
X = df[features].values
y = df[target].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
#
# Tipo de las columnas
#
df.dtypes

In [ ]:
#
# Modelado
#
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Lasso
from sklearn.metrics import r2_score, mean_squared_error

pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("lasso", Lasso(max_iter=10000)),  # alpha=1.0
    ]
)

pipeline.fit(X_train, y_train)

y_train_pred = pipeline.predict(X_train)
y_test_pred = pipeline.predict(X_test)


rsquared_train = r2_score(y_train, y_train_pred)
rsquared_test = r2_score(y_test, y_test_pred)

mse_train = mean_squared_error(y_train, y_train_pred)
mse_test = mean_squared_error(y_test, y_test_pred)


print(f"Mean Squared Error:    {mse_train:0.2f} ({mse_test:0.2f})")
print(f"         R-squared: {rsquared_train:6.5f} ({rsquared_test:6.5f})")

In [ ]:
#
# Análisis del alpha
#
import numpy as np

coeff = pd.Series(pipeline["lasso"].coef_, index=features)

alphas = np.linspace(0.01, 1000, 100)
coefs = []

for a in alphas:
    pipeline["lasso"].set_params(alpha=a)
    pipeline.fit(X_train, y_train)
    coefs.append(pipeline["lasso"].coef_)

ax = plt.gca()
ax.plot(alphas * 2, coefs)
ax.set_xscale("log")
ax.legend(features)
ax.grid(False)
plt.axis("tight")
plt.xlabel("alpha")
plt.ylabel("coefficients")
plt.title("Coefficient values with increasing values of alpha")

In [ ]:
# Se selecciona alpha mediante validación cruzada en entrenamiento.

from sklearn.model_selection import GridSearchCV

alphas = {"lasso__alpha": np.linspace(0.01, 2.0, 100)}

grid_search = GridSearchCV(
    pipeline,
    alphas,
    scoring="neg_mean_squared_error",
    cv=10,
)

grid_search.fit(X_train, y_train)

y_train_pred = grid_search.predict(X_train)
y_test_pred = grid_search.predict(X_test)

rsquared_train = r2_score(y_train, y_train_pred)
rsquared_test = r2_score(y_test, y_test_pred)

mse_train = mean_squared_error(y_train, y_train_pred)
mse_test = mean_squared_error(y_test, y_test_pred)

print(f"     Best value for lambda : ", grid_search.best_params_)
print("Best score for cost function: ", grid_search.best_score_)
print()
print(f"Mean Squared Error:    {mse_train:0.2f} ({mse_test:0.2f})")
print(f"         R-squared: {rsquared_train:6.5f} ({rsquared_test:6.5f})")

In [ ]:
#
# Salva el modelo
#
import pickle
import os


with open("../submission/estimator.pkl", "wb") as f:
    pickle.dump(grid_search, f)